In [ ]:
import numpy as np
import numpy.random as rnd
import numpy.linalg as linalg
import math
from scipy.linalg import sqrtm
from scipy.optimize import minimize
import sys

Functions for the State Evolution loop functions

In [ ]:
def Softplus(x):
    return np.log(1 + np.exp(x))

def DSoftplus(x):
    return 1/(1 + np.exp(-x))

def DDSoftplus(x):
    return DSoftplus(x)*np.exp(-x)/(1 + np.exp(-x))

def Loss(y, z):
    return ((y - z[0])*(y - z[0])/(2*Softplus(z[1])) + np.log(Softplus(z[1])))/2

def DDzLoss(y, z):
    Sz = Softplus(z[1])
    Dsz = DSoftplus(z[1])
    return np.array([[1/Sz, (y - z[0])*Dsz/(Sz*Sz)],[(y - z[0])*Dsz/(Sz*Sz), (Sz*DDSoftplus(z[1])*(Sz - (y - z[0])*(y - z[0])) - 2*Dsz*Dsz*(Sz/2 - (y - z[0])*(y - z[0])))/(2*Sz*Sz*Sz)]])

def Prox(mu, Omega, f):
    ToOptimize = lambda x: (np.transpose(x - mu) @ linalg.inv(Omega) @ (x - mu))/2 + f(x)
    Prox = minimize(ToOptimize, x0=[0, 0])
    return Prox.x

def T(mhat, qhat, chihat, W, xi):
    sqrtqhat = sqrtm(qhat).real
    return linalg.inv(chihat) @ (mhat @ W + sqrtqhat @ xi)

def fw(R, SigmaInv, Lambda):
    return linalg.inv(SigmaInv + Lambda*np.eye(2)) @ SigmaInv @ R

def fc(SigmaInv, Lambda):
    return linalg.inv(SigmaInv) @ linalg.inv(SigmaInv + Lambda*np.eye(2)) @ SigmaInv

def phi(z, A):
    return z[0] + A*np.sqrt(Softplus(z[1]))

def Dz_phi(z, A):
    return np.array([1, np.divide(A*DSoftplus(z[1]), 2*np.sqrt(Softplus(z[1])))])

def gout(zStar, w, V):
    return linalg.inv(V) @ (zStar - w)

def Domega_gout(zStar, y, V):
    InvV = linalg.inv(V)
    return InvV @ (linalg.inv(InvV + DDzLoss(y, zStar)) @ InvV - np.eye(2))

def Dy_gout(zStar, y, V):
    InvV = linalg.inv(V)
    Sz = Softplus(zStar[1])
    Dsz = DSoftplus(zStar[1])
    return InvV @ (linalg.inv(InvV + DDzLoss(y, zStar)) @ np.array([1/Sz, (Dsz*(y - zStar[0]))/(Sz*Sz)]))


State Evolution Loop functions

In [ ]:
def RealFuncs(mhat, qhat, chihat, W, xi, Lambda):
    fwval = fw(T(mhat, qhat, chihat, W, xi), chihat, Lambda)
    m = fwval @ np.transpose(W)
    q = fwval @ np.transpose(fwval)
    sigma = fc(chihat, Lambda)
    return m, q, sigma

def HatFuncs(sigma, z, w, A):
    y = phi(z, A)
    Optimizedz = Prox(w, sigma, lambda z: Loss(y, z))
    qhat = gout(y, w, sigma) @ np.transpose(gout(y, w, sigma))
    mhat = Dy_gout(Optimizedz, y, sigma) @ Dz_phi(z, A)
    chihat = Domega_gout(Optimizedz, y, sigma)
    return qhat, mhat, chihat

State Evolution sampling functions

In [ ]:
def TrueRandSampleReal():
    W = rnd.normal(0, 1, 2)
    xi = rnd.normal(0, 1, 2)
    return W, xi

def TrueRandSampleHat(q, m):
    zw0 = rnd.multivariate_normal([0, 0], [[1, m[0,0]],[m[0,0], q[0,0]]])
    zw1 = rnd.multivariate_normal([0, 0], [[1, m[1,1]],[m[1,1], q[1,1]]])
    A = rnd.normal(0, 1)
    return np.array([zw0[0], zw1[0]]), np.array([zw0[1], zw1[1]]), A

State Evolution Expectation functions

In [ ]:
def TrueRandExpectReal(mhat, qhat, chihat, Lambda, Nsample):
    m, q, sigma = np.zeros((2, 2)), np.zeros((2, 2)), np.zeros((2, 2))
    for i in range(Nsample):
        W, xi = TrueRandSampleReal()
        newm, newq, newsigma = RealFuncs(mhat, qhat, chihat, W, xi, Lambda)
        m += newm
        q += newq
        sigma += newsigma
    return m/Nsample, q/Nsample, sigma/Nsample

def TrueRandExpectHat(q, m, sigma, alpha, Nsample):
    qhat, mhat, chihat = np.zeros((2, 2)), np.zeros((2, 2)), np.zeros((2, 2))
    for i in range(Nsample):
        z, w, A = TrueRandSampleHat(q, m)
        newqhat, newmhat, newchihat = HatFuncs(sigma, z, w, A)
        qhat += newqhat
        mhat += newmhat
        chihat -= newchihat
    return alpha*qhat/Nsample, alpha*mhat/Nsample, alpha*chihat/Nsample

State Evolution runner

In [ ]:
def TrueRandSE_ERM(alpha, Lambda, q0, m0, sigma0, Nsample = 1000, MaxIter = 1e4, EpsConvergence = 1e-6, Verbose = True, VerboseRate = 1, DebugVerbose = False):
    q, m, sigma = q0, m0, sigma0
    qhat, mhat, chihat = np.eye(2), np.eye(2), np.eye(2)
    NIter = 0
    Conv = 1
    while((Conv > EpsConvergence) and (NIter < MaxIter)):
        qhat, mhat, chihat = TrueRandExpectHat(q, m, sigma, alpha, Nsample)
        newm, newq, newsigma = TrueRandExpectReal(mhat, qhat, chihat, Lambda, Nsample)
        Conv = (np.abs(q[0,0] - newq[0,0]) + np.abs(q[1,1] - newq[1,1]))/(np.abs(newq[0,0]) + np.abs(newq[1,1]))
        m, q, sigma = newm, newq, newsigma
        if(Verbose and NIter%VerboseRate == 0):
            print("Iteration %s" % NIter)
            print("Current convergence criterion %s" % Conv)
        if(DebugVerbose):
            print("qhat", qhat)
            print("mhat", mhat)
            print("chihat", chihat)
            print("m", m)
            print("q", q)
            print("sigma", sigma)
        NIter += 1
    return q, m, sigma

Main

In [ ]:
alpha = 10
Lambda = 1
q0 = np.array([[0.3, 0.4],[0.3, 0.2]])#0.5*np.eye(2)
m0 = np.array([[0.1, 0.2],[0.1, 0.3]])#0.5*np.eye(2)
sigma0 = 0.5*np.eye(2)
TrueRandSE_ERM(alpha, Lambda, q0, m0, sigma0, Nsample = 5000, DebugVerbose = True)